# Limpeza e tratamento dos dados do TSE referente ao estado de MG

In [9]:
import pandas as pd
import os
import numpy as np
import unicodedata

In [10]:
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return texto
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto.upper().replace('-', ' ').strip()

In [11]:
municipios_alvo = ['SAO JOAO DEL REI', 'DIVINOPOLIS', 'OURO BRANCO', 'SETE LAGOAS']

arquivo_origem = "votacao_secao_2026_MG.csv"
chunk_size = 100000

colunas_int = [
    'ANO_ELEICAO', 'CD_TIPO_ELEICAO', 'NR_TURNO', 'CD_ELEICAO', 
    'CD_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
    'NR_VOTAVEL', 'QT_VOTOS', 'NR_LOCAL_VOTACAO', 'SQ_CANDIDATO'
]

colunas_datas = ['DT_GERACAO', 'DT_ELEICAO']

valores_nulos = ['#NULO', '#NE', '#NULO#', '#NE#', -1, -3, '-1', '-3']

In [12]:
for chunk in pd.read_csv(arquivo_origem, encoding='latin1', sep=';', chunksize=chunk_size, low_memory=False):
    chunk = chunk.replace(valores_nulos, np.nan)
    chunk['NM_MUNICIPIO_CLEAN'] = chunk['NM_MUNICIPIO'].astype(str).apply(normalizar_texto)
    
    # Conversão de Tipos de Dados
    for col in colunas_datas:
        if col in chunk.columns:
            chunk[col] = pd.to_datetime(chunk[col], format='%d/%m/%Y', errors='coerce')
            
    for col in colunas_int:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors='coerce').astype('Int64')
            
    # Separar arquivos por municipio
    for mun in municipios_alvo:
        df_mun = chunk[chunk['NM_MUNICIPIO_CLEAN'] == mun].copy()
        
        if not df_mun.empty:
            df_mun = df_mun.drop(columns=['NM_MUNICIPIO_CLEAN'])
            
            nome_slug = mun.lower().replace(' ', '_')
            nome_arquivo = f"votacao_{nome_slug}.csv"
            
            header = not os.path.exists(nome_arquivo)
            df_mun.to_csv(nome_arquivo, mode='a', index=False, sep=';', encoding='utf-8-sig', header=header)
print("Arquivos salvos!")

Arquivos salvos!
